# Análisis de Ventas con PySpark

**Proyecto:** Análisis de ventas combinando datos transaccionales con información de clientes

**Objetivo:** Realizar limpieza, filtrado y análisis agregado de datos de ventas usando PySpark SQL

---

## 1. Instalación e Imports

Instalamos PySpark e importamos las librerías necesarias.

In [19]:
# Instalar PySpark (descomentar si es necesario)
# !pip install pyspark

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, sum as _sum, count, avg, max as _max, min as _min, regexp_replace
from pathlib import Path
import sys

## 2. Creación de SparkSession

Iniciamos Spark con configuración local para desarrollo.

In [20]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Analisis_Ventas") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark Session creada exitosamente")

Spark Version: 4.1.1
Spark Session creada exitosamente


## 3. Búsqueda Automática de la Carpeta de Datos

Función para localizar la carpeta `datos_proyecto` desde el directorio actual o directorios padres.

In [21]:
def find_data_dir(dirname: str = "datos_proyecto") -> Path:
    """
    Busca la carpeta de datos navegando desde el directorio actual hacia arriba.
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / dirname
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontró la carpeta '{dirname}' desde {Path.cwd()}")

# Buscar carpeta de datos
data_dir = find_data_dir()
print(f"Carpeta de datos encontrada en: {data_dir}")

Carpeta de datos encontrada en: /Users/diegocastro/Desktop/upb_big_data/datos_proyecto


## 4. Carga de Archivos CSV

Cargamos los dos archivos CSV con delimitador `;`:
- **export.csv**: Datos transaccionales de ventas
- **maestro_clientes.csv**: Datos maestros de clientes

In [22]:
# Configuración de lectura CSV
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "delimiter": ";"
}

# Cargar archivo de ventas (FACT)
df_ventas = spark.read.options(**csv_options).csv(str(data_dir / "export.csv"))
print(f"Ventas cargadas: {df_ventas.count()} registros")

# Cargar archivo de clientes (DIM)
df_clientes = spark.read.options(**csv_options).csv(str(data_dir / "maestro_clientes.csv"))
print(f"Clientes cargados: {df_clientes.count()} registros")

Ventas cargadas: 5638 registros
Clientes cargados: 282568 registros


## 5. Exploración Inicial de Datos

Analizamos la estructura y contenido de ambos DataFrames.

In [23]:
# Esquema de ventas
print("=== ESQUEMA VENTAS ===")
df_ventas.printSchema()

=== ESQUEMA VENTAS ===
root
 |-- Market: string (nullable = true)
 |-- Clase de coste: string (nullable = true)
 |-- Período/Año: double (nullable = true)
 |-- Brand family: string (nullable = true)
 |-- Artículo: string (nullable = true)
 |-- Production Center: string (nullable = true)
 |-- Unidad Base UoM: string (nullable = true)
 |-- Qty in Base UoM: string (nullable = true)
 |-- GROSS SALES -TRADE: string (nullable = true)
 |-- EXCISE TAXES: string (nullable = true)
 |-- OTHER SVC: string (nullable = true)
 |-- COS-PURC. CONS. AFF: string (nullable = true)
 |-- COS - SVC CONV. COST: string (nullable = true)
 |-- Moneda: string (nullable = true)
 |-- Nº documento: integer (nullable = true)
 |-- EU AGRREMENT - SEIZED PROD COSTS (BELOW: string (nullable = true)
 |-- Documento anulado: integer (nullable = true)
 |-- Posic.docum.anulada: string (nullable = true)
 |-- Nº documento refer.: long (nullable = true)
 |-- Fecha de factura: string (nullable = true)
 |-- Clase de factura: strin

In [24]:
# Muestra de ventas
print("\n=== MUESTRA VENTAS ===")
df_ventas.show(5, truncate=False)


=== MUESTRA VENTAS ===
+------+--------------+-----------+------------+-----------+-----------------+---------------+---------------+------------------+------------+---------+-------------------+--------------------+------+------------+---------------------------------------+-----------------+-------------------+-------------------+----------------+----------------+------------------+-------+
|Market|Clase de coste|Período/Año|Brand family|Artículo   |Production Center|Unidad Base UoM|Qty in Base UoM|GROSS SALES -TRADE|EXCISE TAXES|OTHER SVC|COS-PURC. CONS. AFF|COS - SVC CONV. COST|Moneda|Nº documento|EU AGRREMENT - SEIZED PROD COSTS (BELOW|Documento anulado|Posic.docum.anulada|Nº documento refer.|Fecha de factura|Clase de factura|Canal distribución|Cliente|
+------+--------------+-----------+------------+-----------+-----------------+---------------+---------------+------------------+------------+---------+-------------------+--------------------+------+------------+-----------------

In [25]:
# Esquema de clientes
print("\n=== ESQUEMA CLIENTES ===")
df_clientes.printSchema()


=== ESQUEMA CLIENTES ===
root
 |-- _c0: string (nullable = true)
 |-- Cliente: integer (nullable = true)
 |-- CanalSap: integer (nullable = true)
 |-- Regional: string (nullable = true)
 |-- Estatus: string (nullable = true)
 |-- NombreCliente: string (nullable = true)
 |-- NombreComercial: string (nullable = true)
 |-- ConcBusq: string (nullable = true)
 |-- Ruta: string (nullable = true)
 |-- Ejecutivos: string (nullable = true)
 |-- Territorio: string (nullable = true)
 |-- EjecutivoTerritorio: string (nullable = true)
 |-- Supervisor   : string (nullable = true)
 |-- DireccionEntrega: string (nullable = true)
 |-- CorreoElectronico: string (nullable = true)
 |-- DescDescuento: string (nullable = true)
 |-- Poblacion: string (nullable = true)
 |-- DescRegion: string (nullable = true)
 |-- Telefono1: string (nullable = true)
 |-- Telefono2: string (nullable = true)
 |-- ID: string (nullable = true)
 |-- PEPCE: string (nullable = true)
 |-- Barrio: string (nullable = true)
 |-- Clase

In [26]:
# Muestra de clientes
print("\n=== MUESTRA CLIENTES ===")
df_clientes.show(5, truncate=False)


=== MUESTRA CLIENTES ===
+----+-------+--------+-----------+-------+--------------------------------+--------------------+------------+-------+-----------------+----------+-------------------+---------------+--------------------+---------------------------+-------------+----------------+---------------+----------+---------+--------+-----+---------------+-------------+--------------------+--------+---------------+-----------------+-------------+---------------------+---------------+--------+------------+-------------+------------+------------------+--------------+------+--------------+-----------+-----------+----------------+-------------+-------------------------+
|_c0 |Cliente|CanalSap|Regional   |Estatus|NombreCliente                   |NombreComercial     |ConcBusq    |Ruta   |Ejecutivos       |Territorio|EjecutivoTerritorio|Supervisor     |DireccionEntrega    |CorreoElectronico          |DescDescuento|Poblacion       |DescRegion     |Telefono1 |Telefono2|ID      |PEPCE|Barrio     

26/03/18 18:08:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Cliente, CanalSap, Regional, Estatus, NombreCliente, NombreComercial, ConcBusq, Ruta, Ejecutivos, Territorio, EjecutivoTerritorio, Supervisor   , DireccionEntrega, CorreoElectronico, DescDescuento, Poblacion, DescRegion, Telefono1, Telefono2, ID, PEPCE, Barrio, ClaseImpuesto, PborAreaSeleccionada, Creado, PeticionBorrado, BloqueoFinanciero, BloqueoPedido, BqPedAreaSeleccionada, CondicionesPago, ViasPago, GrupoPrecios, Canal, Segmentacion, Subcanal, CuentaAsociada, Bodega, RegistroRentas, CupoCredito, ClaseRiesgo, CodigoFacturador, CodigoPagador, ProteccionDatosPersonales
 Schema: _c0, Cliente, CanalSap, Regional, Estatus, NombreCliente, NombreComercial, ConcBusq, Ruta, Ejecutivos, Territorio, EjecutivoTerritorio, Supervisor   , DireccionEntrega, CorreoElectronico, DescDescuento, Poblacion, DescRegion, Telefono1, Telefono2, ID, PEPCE, Barrio, ClaseImpuesto, PborAreaSeleccionada, Creado, Petici

## 6. Definición de Tabla FACT y DIM

### Selección de Archivos

**Justificación de la selección:**
- **Tabla FACT (export.csv):** Contiene datos transaccionales de ventas con métricas como cantidad vendida, valores de ventas, impuestos y costos. Representa hechos del negocio que ocurren en el tiempo.
- **Tabla DIM (maestro_clientes.csv):** Contiene información descriptiva de clientes como nombre, dirección, región, canal de distribución, segmentación. Es información maestra que complementa las transacciones.

**Relación entre tablas:** Ambas tablas se relacionan mediante el campo `Cliente` (código de cliente).

**Valor del análisis:** Al combinar estas tablas podremos:
- Analizar ventas por región, canal y segmentación de clientes
- Identificar patrones de compra por tipo de cliente
- Evaluar el desempeño comercial por territorio y ejecutivo

## 7. Limpieza de Datos

### Operaciones de Limpieza:
1. **Eliminar registros con valores nulos en campos críticos**
2. **Eliminar duplicados**
3. **Normalizar nombres de columnas para el JOIN**

### Justificación:
- Los valores nulos en campos clave (Cliente, ventas) impedirían análisis correctos
- Los duplicados distorsionarían agregaciones y métricas
- La normalización asegura el JOIN exitoso entre tablas

In [27]:
# LIMPIEZA VENTAS
print("=== LIMPIEZA TABLA VENTAS ===")
print(f"Registros originales: {df_ventas.count()}")

# Eliminar registros con Cliente nulo o GROSS SALES nulo
df_ventas_clean = df_ventas.na.drop(subset=["Cliente", "GROSS SALES -TRADE"])
print(f"Después de eliminar nulos en campos críticos: {df_ventas_clean.count()}")

# Eliminar duplicados
df_ventas_clean = df_ventas_clean.dropDuplicates()
print(f"Después de eliminar duplicados: {df_ventas_clean.count()}")

# Guardar una base limpia antes de convertir tipos para poder reejecutar celdas sin arrastrar casts previos
df_ventas_base_clean = df_ventas_clean

# Mostrar estadísticas
registros_eliminados = df_ventas.count() - df_ventas_clean.count()
porcentaje_eliminado = (registros_eliminados / df_ventas.count()) * 100
print(f"Registros eliminados: {registros_eliminados} ({porcentaje_eliminado:.2f}%)")

=== LIMPIEZA TABLA VENTAS ===
Registros originales: 5638
Después de eliminar nulos en campos críticos: 5638
Después de eliminar duplicados: 5638
Registros eliminados: 0 (0.00%)


In [28]:
# LIMPIEZA CLIENTES
print("\n=== LIMPIEZA TABLA CLIENTES ===")
print(f"Registros originales: {df_clientes.count()}")

# Eliminar registros con Cliente nulo
df_clientes_clean = df_clientes.na.drop(subset=["Cliente"])
print(f"Después de eliminar nulos en Cliente: {df_clientes_clean.count()}")

# Rellenar valores nulos en campos descriptivos con 'NO ESPECIFICADO'
df_clientes_clean = df_clientes_clean.na.fill({
    "Regional": "NO ESPECIFICADO",
    "Segmentacion": "NO ESPECIFICADO",
    "Canal": "NO ESPECIFICADO"
})

# Eliminar duplicados basados en Cliente
df_clientes_clean = df_clientes_clean.dropDuplicates(["Cliente"])
print(f"Después de eliminar duplicados: {df_clientes_clean.count()}")

registros_eliminados_cli = df_clientes.count() - df_clientes_clean.count()
porcentaje_eliminado_cli = (registros_eliminados_cli / df_clientes.count()) * 100
print(f"Registros eliminados: {registros_eliminados_cli} ({porcentaje_eliminado_cli:.2f}%)")


=== LIMPIEZA TABLA CLIENTES ===
Registros originales: 282568
Después de eliminar nulos en Cliente: 282568


Después de eliminar duplicados: 281754
Registros eliminados: 814 (0.29%)


In [29]:
# CONVERSIÓN DE TIPOS DE DATOS (Ventas)
from pyspark.sql.functions import regexp_replace, col, when

print("=== CONVERSIÓN DE TIPOS DE DATOS ===")

# Convertir columnas numéricas que usan coma como separador decimal
numeric_cols = [
    "Qty in Base UoM", 
    "GROSS SALES -TRADE", 
    "EXCISE TAXES", 
    "OTHER SVC",
    "COS-PURC. CONS. AFF", 
    "COS - SVC CONV. COST",
    "EU AGRREMENT - SEIZED PROD COSTS (BELOW"
]

# Rehacer la conversión siempre desde la base limpia sin casts previos
df_ventas_clean = df_ventas_base_clean

for col_name in numeric_cols:
    cleaned_col = when(
        col(f"`{col_name}`").contains(","),
        regexp_replace(
            regexp_replace(col(f"`{col_name}`"), "\\.", ""),
            ",",
            "."
        )
    ).otherwise(col(f"`{col_name}`"))

    df_ventas_clean = df_ventas_clean.withColumn(
        col_name,
        cleaned_col.cast("double")
    )

print("Columnas numéricas convertidas exitosamente")
print("\nEsquema actualizado:")
df_ventas_clean.select(*[col(f"`{col_name}`") for col_name in numeric_cols]).printSchema()

=== CONVERSIÓN DE TIPOS DE DATOS ===
Columnas numéricas convertidas exitosamente

Esquema actualizado:
root
 |-- Qty in Base UoM: double (nullable = true)
 |-- GROSS SALES -TRADE: double (nullable = true)
 |-- EXCISE TAXES: double (nullable = true)
 |-- OTHER SVC: double (nullable = true)
 |-- COS-PURC. CONS. AFF: double (nullable = true)
 |-- COS - SVC CONV. COST: double (nullable = true)
 |-- EU AGRREMENT - SEIZED PROD COSTS (BELOW: double (nullable = true)



# FILTRADO VENTAS
print("=== FILTRADO TABLA VENTAS ===")
print(f"Registros antes del filtro: {df_ventas_clean.count()}")

# Filtrar ventas del período 3.2026 (valor double) y con GROSS SALES positivo
df_ventas_filtered = df_ventas_clean.filter(
    (col("Período/Año") == 3.2026) & 
    (col("GROSS SALES -TRADE") > 0)
)
print(f"Después del filtro: {df_ventas_filtered.count()}")

registros_filtrados = df_ventas_clean.count() - df_ventas_filtered.count()
porcentaje_filtrado = (registros_filtrados / df_ventas_clean.count()) * 100
print(f"Registros filtrados: {registros_filtrados} ({porcentaje_filtrado:.2f}%)")

In [30]:
# FILTRADO VENTAS
print("=== FILTRADO TABLA VENTAS ===")
print(f"Registros antes del filtro: {df_ventas_clean.count()}")

# Filtrar ventas del período 003.2026 y con GROSS SALES positivo
df_ventas_filtered = df_ventas_clean.filter(
    (col("Período/Año") == "003.2026") & 
    (col("GROSS SALES -TRADE") > 0)
)
print(f"Después del filtro: {df_ventas_filtered.count()}")

registros_filtrados = df_ventas_clean.count() - df_ventas_filtered.count()
porcentaje_filtrado = (registros_filtrados / df_ventas_clean.count()) * 100
print(f"Registros filtrados: {registros_filtrados} ({porcentaje_filtrado:.2f}%)")

=== FILTRADO TABLA VENTAS ===
Registros antes del filtro: 5638
Después del filtro: 4734
Registros filtrados: 904 (16.03%)


In [31]:
# FILTRADO CLIENTES
print("\n=== FILTRADO TABLA CLIENTES ===")
print(f"Registros antes del filtro: {df_clientes_clean.count()}")

# Filtrar solo clientes activos
df_clientes_filtered = df_clientes_clean.filter(col("Estatus") == "ACTIVO")
print(f"Después del filtro: {df_clientes_filtered.count()}")

registros_filtrados_cli = df_clientes_clean.count() - df_clientes_filtered.count()
porcentaje_filtrado_cli = (registros_filtrados_cli / df_clientes_clean.count()) * 100
print(f"Registros filtrados: {registros_filtrados_cli} ({porcentaje_filtrado_cli:.2f}%)")


=== FILTRADO TABLA CLIENTES ===
Registros antes del filtro: 281754


Después del filtro: 64078


Registros filtrados: 217676 (77.26%)


## 9. Creación de Vistas Temporales y JOIN

Creamos vistas temporales para usar SQL de PySpark y realizamos el JOIN entre ventas y clientes.

In [32]:
# Crear vistas temporales
df_ventas_filtered.createOrReplaceTempView("ventas")
df_clientes_filtered.createOrReplaceTempView("clientes")

print("Vistas temporales creadas: 'ventas' y 'clientes'")

Vistas temporales creadas: 'ventas' y 'clientes'


In [34]:
# JOIN entre ventas y clientes
query_join = """
SELECT 
    v.Cliente,
    c.NombreCliente,
    c.Regional,
    c.Canal,
    c.Segmentacion,
    c.Subcanal,
    c.DescRegion,
    c.Poblacion,
    v.`Brand family` AS Marca,
    v.`Artículo` AS Articulo,
    v.`Qty in Base UoM` AS Cantidad,
    v.`GROSS SALES -TRADE` AS VentaBruta,
    v.`EXCISE TAXES` AS Impuestos,
    v.`COS-PURC. CONS. AFF` AS Costo,
    v.`Fecha de factura` AS FechaFactura,
    v.`Canal distribución` AS CanalDistribucion
FROM ventas v
INNER JOIN clientes c ON v.Cliente = c.Cliente
"""

df_ventas_enriquecidas = spark.sql(query_join)
print(f"Registros después del JOIN: {df_ventas_enriquecidas.count()}")
df_ventas_enriquecidas.show(10)

Registros después del JOIN: 4718


+-------+--------------------+-----------+------------+----------------+------------------+----------+---------+-----+-----------+--------+----------+---------+-----+------------+-----------------+
|Cliente|       NombreCliente|   Regional|       Canal|    Segmentacion|          Subcanal|DescRegion|Poblacion|Marca|   Articulo|Cantidad|VentaBruta|Impuestos|Costo|FechaFactura|CanalDistribucion|
+-------+--------------------+-----------+------------+----------------+------------------+----------+---------+-----+-----------+--------+----------+---------+-----+------------+-----------------+
| 382455|INDUSTRIA CAFETER...|REGIONAL II|DISTRIBUIDOR|SIN SEGMENTACION|DISTRIBUIDOR (EZD)|    NARIÑO|    PASTO|TEREA|ME007962.01|     3.0|     391.3|   118.89|65.31|      5/3/26|               11|
| 382455|INDUSTRIA CAFETER...|REGIONAL II|DISTRIBUIDOR|SIN SEGMENTACION|DISTRIBUIDOR (EZD)|    NARIÑO|    PASTO|  VNU|MA001799.01|    0.01|     54.35|      0.0|18.92|     12/3/26|               11|
| 382455|I

In [35]:
# Crear vista de datos enriquecidos
df_ventas_enriquecidas.createOrReplaceTempView("ventas_enriquecidas")
print("Vista temporal 'ventas_enriquecidas' creada exitosamente")

Vista temporal 'ventas_enriquecidas' creada exitosamente


## 10. Operaciones SQL de Valor Agregado

Realizamos múltiples operaciones analíticas usando SQL de PySpark.

### 10.1 Agregación: Ventas por Regional

**Valor:** Identificar qué regionales generan más ingresos y transacciones.

In [36]:
query_regional = """
SELECT 
    Regional,
    COUNT(*) AS NumTransacciones,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    ROUND(AVG(VentaBruta), 2) AS VentaPromedio,
    ROUND(SUM(Impuestos), 2) AS ImpuestosTotal,
    ROUND(SUM(Costo), 2) AS CostoTotal,
    ROUND(SUM(VentaBruta) - SUM(Costo), 2) AS MargenBruto
FROM ventas_enriquecidas
GROUP BY Regional
ORDER BY VentaTotal DESC
"""

resultado_regional = spark.sql(query_regional)
print("=== VENTAS POR REGIONAL ===")
resultado_regional.show(20, truncate=False)

=== VENTAS POR REGIONAL ===


+------------+----------------+----------+-------------+--------------+----------+-----------+
|Regional    |NumTransacciones|VentaTotal|VentaPromedio|ImpuestosTotal|CostoTotal|MargenBruto|
+------------+----------------+----------+-------------+--------------+----------+-----------+
|KEY ACCOUNTS|4083            |574642.04 |140.74       |147069.77     |113750.96 |460891.08  |
|REGIONAL III|262             |92347.15  |352.47       |27189.23      |17358.81  |74988.34   |
|REGIONAL I  |141             |39157.08  |277.71       |2281.33       |12434.62  |26722.46   |
|REGIONAL II |200             |29479.62  |147.4        |8081.67       |5735.32   |23744.3    |
|REGIONAL IV |32              |3715.34   |116.1        |931.44        |745.75    |2969.59    |
+------------+----------------+----------+-------------+--------------+----------+-----------+



### 10.2 Agregación con Subconsulta: Top Marcas por Segmentación

**Valor:** Identificar qué marcas son más exitosas en cada segmentación de cliente.

In [37]:
query_top_marcas = """
WITH ventas_por_marca AS (
    SELECT 
        Segmentacion,
        Marca,
        ROUND(SUM(VentaBruta), 2) AS VentaTotal,
        COUNT(*) AS NumTransacciones
    FROM ventas_enriquecidas
    WHERE Segmentacion != 'NO ESPECIFICADO'
    GROUP BY Segmentacion, Marca
),
ranking_marcas AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY Segmentacion ORDER BY VentaTotal DESC) AS ranking
    FROM ventas_por_marca
)
SELECT 
    Segmentacion,
    Marca,
    VentaTotal,
    NumTransacciones,
    ranking
FROM ranking_marcas
WHERE ranking <= 3
ORDER BY Segmentacion, ranking
"""

resultado_top_marcas = spark.sql(query_top_marcas)
print("=== TOP 3 MARCAS POR SEGMENTACIÓN ===")
resultado_top_marcas.show(50, truncate=False)

=== TOP 3 MARCAS POR SEGMENTACIÓN ===


+----------------+-----+----------+----------------+-------+
|Segmentacion    |Marca|VentaTotal|NumTransacciones|ranking|
+----------------+-----+----------+----------------+-------+
|1A              |TEREA|340466.13 |1546            |1      |
|1A              |ZYN  |24823.34  |370             |2      |
|1A              |VEEV |7364.98   |243             |3      |
|1B              |TEREA|6793.9    |67              |1      |
|1C              |TEREA|1504.94   |28              |1      |
|2A              |TEREA|3785.69   |53              |1      |
|2A              |ZYN  |309.26    |7               |2      |
|2A              |VNU  |58.9      |2               |3      |
|2B              |TEREA|2587.92   |46              |1      |
|2B              |VEEV |69.72     |3               |2      |
|2B              |VNU  |41.61     |3               |3      |
|2C              |TEREA|5358.63   |82              |1      |
|3A              |TEREA|313.95    |3               |1      |
|3B              |TEREA|

### 10.3 Función de Ventana: Ranking de Clientes por Ventas

**Valor:** Identificar los mejores clientes usando funciones de ventana (ROW_NUMBER, DENSE_RANK).

In [38]:
query_ranking_clientes = """
SELECT 
    Cliente,
    NombreCliente,
    Regional,
    Segmentacion,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    COUNT(*) AS NumTransacciones,
    ROUND(AVG(VentaBruta), 2) AS VentaPromedio,
    ROW_NUMBER() OVER (ORDER BY SUM(VentaBruta) DESC) AS RankingGeneral,
    DENSE_RANK() OVER (PARTITION BY Regional ORDER BY SUM(VentaBruta) DESC) AS RankingRegional,
    ROUND(SUM(VentaBruta) * 100.0 / SUM(SUM(VentaBruta)) OVER (), 4) AS PorcentajeVentaTotal
FROM ventas_enriquecidas
GROUP BY Cliente, NombreCliente, Regional, Segmentacion
ORDER BY VentaTotal DESC
LIMIT 20
"""

resultado_ranking = spark.sql(query_ranking_clientes)
print("=== RANKING TOP 20 CLIENTES ===")
resultado_ranking.show(20, truncate=False)

=== RANKING TOP 20 CLIENTES ===


26/03/18 18:09:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:09:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:09:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:09:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:09:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:09:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 1

+-------+--------------------------------------+------------+----------------+----------+----------------+-------------+--------------+---------------+--------------------+
|Cliente|NombreCliente                         |Regional    |Segmentacion    |VentaTotal|NumTransacciones|VentaPromedio|RankingGeneral|RankingRegional|PorcentajeVentaTotal|
+-------+--------------------------------------+------------+----------------+----------+----------------+-------------+--------------+---------------+--------------------+
|1473666|CADENA COMERCIAL OXXO COLOMBIA S.A.S  |KEY ACCOUNTS|1A              |113580.26 |19              |5977.91      |1             |1              |15.3624             |
|1843664|OROZCO MONTOYA JUAN DIEGO             |REGIONAL III|Diamond +       |26519.4   |5               |5303.88      |2             |1              |3.5869              |
|1726642|MARKETING TOOLS AGENCY S.A.S          |REGIONAL I  |SIN SEGMENTACION|25772.58  |24              |1073.86      |3             |

### 10.4 ROLLUP: Análisis Jerárquico de Ventas

**Valor:** Obtener subtotales a diferentes niveles de agregación (Regional > Canal > Subcanal).

In [39]:
query_rollup = """
SELECT 
    COALESCE(Regional, 'TOTAL GENERAL') AS Regional,
    COALESCE(Canal, 'TOTAL REGIONAL') AS Canal,
    COALESCE(Subcanal, 'TOTAL CANAL') AS Subcanal,
    COUNT(*) AS NumTransacciones,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    ROUND(AVG(VentaBruta), 2) AS VentaPromedio
FROM ventas_enriquecidas
WHERE Regional != 'NO ESPECIFICADO'
GROUP BY Regional, Canal, Subcanal WITH ROLLUP
ORDER BY Regional NULLS LAST, Canal NULLS LAST, Subcanal NULLS LAST
"""

resultado_rollup = spark.sql(query_rollup)
print("=== ANÁLISIS JERÁRQUICO CON ROLLUP ===")
resultado_rollup.show(50, truncate=False)

=== ANÁLISIS JERÁRQUICO CON ROLLUP ===


+-------------+-----------------+-----------------------------------------------+----------------+----------+-------------+
|Regional     |Canal            |Subcanal                                       |NumTransacciones|VentaTotal|VentaPromedio|
+-------------+-----------------+-----------------------------------------------+----------------+----------+-------------+
|KEY ACCOUNTS |KEY ACCOUNTS - KA|ALMACÉN DE CADENA NACIONAL                     |4083            |574642.04 |140.74       |
|KEY ACCOUNTS |KEY ACCOUNTS - KA|TOTAL CANAL                                    |4083            |574642.04 |140.74       |
|KEY ACCOUNTS |TOTAL REGIONAL   |TOTAL CANAL                                    |4083            |574642.04 |140.74       |
|REGIONAL I   |CONSUMO LOCAL    |BAR / DISCOTECA                                |31              |26250.87  |846.8        |
|REGIONAL I   |CONSUMO LOCAL    |CAFETERÍA / RESTAURANTE / PAPELERÍA /MISCELÁNEA|3               |421.43    |140.48       |
|REGIONA

### 10.5 Operación de Conjunto: UNION ALL - Comparación de Canales

**Valor:** Combinar análisis de diferentes canales en un solo resultado.

In [40]:
query_union = """
SELECT 
    'TRADICIONALES' AS TipoCanal,
    COUNT(*) AS NumTransacciones,
    COUNT(DISTINCT Cliente) AS NumClientes,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    ROUND(AVG(VentaBruta), 2) AS TicketPromedio
FROM ventas_enriquecidas
WHERE Canal = 'TRADICIONALES'

UNION ALL

SELECT 
    'MAYORISTA' AS TipoCanal,
    COUNT(*) AS NumTransacciones,
    COUNT(DISTINCT Cliente) AS NumClientes,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    ROUND(AVG(VentaBruta), 2) AS TicketPromedio
FROM ventas_enriquecidas
WHERE Canal = 'MAYORISTA'

UNION ALL

SELECT 
    'OTROS' AS TipoCanal,
    COUNT(*) AS NumTransacciones,
    COUNT(DISTINCT Cliente) AS NumClientes,
    ROUND(SUM(VentaBruta), 2) AS VentaTotal,
    ROUND(AVG(VentaBruta), 2) AS TicketPromedio
FROM ventas_enriquecidas
WHERE Canal NOT IN ('TRADICIONALES', 'MAYORISTA') AND Canal != 'NO ESPECIFICADO'

ORDER BY VentaTotal DESC
"""

resultado_union = spark.sql(query_union)
print("=== COMPARACIÓN DE CANALES (UNION ALL) ===")
resultado_union.show(truncate=False)

=== COMPARACIÓN DE CANALES (UNION ALL) ===


+-------------+----------------+-----------+----------+--------------+
|TipoCanal    |NumTransacciones|NumClientes|VentaTotal|TicketPromedio|
+-------------+----------------+-----------+----------+--------------+
|OTROS        |4544            |645        |651412.39 |143.36        |
|MAYORISTA    |174             |37         |87928.84  |505.34        |
|TRADICIONALES|0               |0          |NULL      |NULL          |
+-------------+----------------+-----------+----------+--------------+



### 10.6 Análisis Avanzado: Pareto de Clientes (Regla 80/20)

**Valor:** Identificar qué clientes generan el 80% de las ventas usando funciones de ventana con suma acumulada.

In [41]:
query_pareto = """
WITH ventas_cliente AS (
    SELECT 
        Cliente,
        NombreCliente,
        Regional,
        ROUND(SUM(VentaBruta), 2) AS VentaTotal
    FROM ventas_enriquecidas
    GROUP BY Cliente, NombreCliente, Regional
),
ventas_acumuladas AS (
    SELECT 
        *,
        SUM(VentaTotal) OVER (ORDER BY VentaTotal DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS VentaAcumulada,
        SUM(VentaTotal) OVER () AS VentaTotalGlobal
    FROM ventas_cliente
)
SELECT 
    Cliente,
    NombreCliente,
    Regional,
    VentaTotal,
    VentaAcumulada,
    ROUND((VentaAcumulada / VentaTotalGlobal) * 100, 2) AS PorcentajeAcumulado,
    CASE 
        WHEN (VentaAcumulada / VentaTotalGlobal) <= 0.80 THEN 'TOP 80%'
        ELSE 'RESTO 20%'
    END AS ClasificacionPareto
FROM ventas_acumuladas
ORDER BY VentaTotal DESC
LIMIT 30
"""

resultado_pareto = spark.sql(query_pareto)
print("=== ANÁLISIS PARETO DE CLIENTES ===")
resultado_pareto.show(30, truncate=False)

=== ANÁLISIS PARETO DE CLIENTES ===


26/03/18 18:10:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 1

+-------+--------------------------------------+------------+----------+------------------+-------------------+-------------------+
|Cliente|NombreCliente                         |Regional    |VentaTotal|VentaAcumulada    |PorcentajeAcumulado|ClasificacionPareto|
+-------+--------------------------------------+------------+----------+------------------+-------------------+-------------------+
|1473666|CADENA COMERCIAL OXXO COLOMBIA S.A.S  |KEY ACCOUNTS|113580.26 |113580.26         |15.36              |TOP 80%            |
|1843664|OROZCO MONTOYA JUAN DIEGO             |REGIONAL III|26519.4   |140099.66         |18.95              |TOP 80%            |
|1726642|MARKETING TOOLS AGENCY S.A.S          |REGIONAL I  |25772.58  |165872.24         |22.44              |TOP 80%            |
|1027997|BOTERO GOMEZ FABIO DE JESUS           |REGIONAL III|24214.26  |190086.5          |25.71              |TOP 80%            |
|1473665|CADENA COMERCIAL OXXO COLOMBIA S.A.S  |KEY ACCOUNTS|16109.92  |2061

26/03/18 18:10:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/18 18:10:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 11. Resultados e Insights

### Resumen de Hallazgos:

1. **Ventas por Regional:**
   - Identificamos qué regionales tienen mejor desempeño en ventas y márgenes
   - Útil para asignación de recursos y estrategias regionales

2. **Top Marcas por Segmentación:**
   - Descubrimos qué marcas tienen mejor aceptación en cada tipo de cliente
   - Permite personalizar estrategias de marketing por segmento

3. **Ranking de Clientes:**
   - Identificamos los clientes más valiosos usando múltiples métricas
   - El ranking regional ayuda a priorizar atención comercial por zona

4. **Análisis Jerárquico (ROLLUP):**
   - Proporcionamos subtotales a diferentes niveles de agregación
   - Facilita el análisis drill-down desde regional hasta subcanal

5. **Comparación de Canales:**
   - Comparamos el desempeño entre canales tradicionales, mayoristas y otros
   - El ticket promedio revela diferentes patrones de compra

6. **Análisis Pareto:**
   - Identificamos qué clientes generan el 80% de las ventas
   - Permite focalizar esfuerzos comerciales en clientes de alto valor

## 12. Conclusión

### Técnicas Aplicadas:
✅ **Limpieza de datos:** Eliminación de nulos, duplicados y normalización

✅ **Filtrado:** Selección de período relevante, ventas positivas y clientes activos

✅ **JOIN:** Integración de datos transaccionales con maestros de clientes

✅ **Agregaciones:** COUNT, SUM, AVG, MIN, MAX por diferentes dimensiones

✅ **Subconsultas:** WITH (CTE) para análisis complejos

✅ **Funciones de ventana:** ROW_NUMBER, DENSE_RANK, SUM acumulado, porcentajes

✅ **ROLLUP:** Análisis jerárquico con subtotales

✅ **UNION ALL:** Combinación de resultados de diferentes segmentos

### Valor del Análisis:
Este notebook proporciona una visión completa del negocio combinando datos transaccionales con información de clientes, permitiendo:
- Identificar oportunidades de crecimiento por región y canal
- Priorizar clientes de alto valor
- Optimizar estrategias de producto por segmentación
- Tomar decisiones basadas en datos con métricas claras y accionables

### Próximos Pasos Sugeridos:
- Análisis temporal de tendencias de ventas
- Segmentación de clientes usando clustering
- Predicción de ventas usando machine learning
- Análisis de churn y retención de clientes

In [42]:
# Detener SparkSession
spark.stop()
print("Spark Session finalizada")

Spark Session finalizada
